# MobilePlantViT Block Validation Notebook

This notebook provides visual verification for all neural network blocks.

## Contents
1. Setup and Imports
2. Block-by-Block Testing
3. Full Pipeline Verification
4. Benchmark Results

In [ ]:
# Setup and Imports
import sys
sys.path.insert(0, '..')  # Add parent directory to path

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np

# Set random seed for reproducibility
torch.manual_seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Utility Imports

Import testing and benchmarking utilities.

In [ ]:
# Import utilities
from src.utils.testing import (
    check_output_shape,
    check_gradient_flow,
    check_no_nan_inf,
    count_parameters,
)

from src.utils.benchmarking import (
    time_forward_backward,
    warmup_and_benchmark,
)

print("Utilities imported successfully!")

## 2. Block Imports

These imports will be uncommented as blocks are implemented.

In [ ]:
# Block imports - uncomment as implemented
# from src.blocks import (
#     GhostConv,
#     CoordAtt,
#     FusedInvertedResidualBlock,
#     PatchEmbedding,
#     PositionalEncoding,
#     LinearDifferentialAttention,
#     BottleneckFFN,
#     ResidualLayerNormBlock,
#     GlobalAveragePooling,
#     ClassifierHead,
# )

print("Block imports ready (uncomment as implemented)")

In [ ]:
# GhostConv Validation
print("=" * 60)
print("GhostConv Validation")
print("=" * 60)

from src.blocks import GhostConv

# Test configurations
configs = [
    {"name": "Basic (3→64)", "params": {"inp": 3, "oup": 64}, "input_shape": (2, 3, 224, 224)},
    {"name": "With stride=2", "params": {"inp": 64, "oup": 128, "stride": 2}, "input_shape": (2, 64, 56, 56)},
    {"name": "Ratio=4", "params": {"inp": 64, "oup": 64, "ratio": 4}, "input_shape": (2, 64, 56, 56)},
]

for cfg in configs:
    ghost = GhostConv(**cfg['params'])
    x = torch.randn(*cfg['input_shape'])
    
    success = validate_block(
        ghost, 
        cfg['input_shape'], 
        None,  # Will check actual output
        cfg['name']
    )
    
    # Show internal channels
    print(f"  Internal: init_channels={ghost.init_channels}, new_channels={ghost.new_channels}")

print("\n✅ GhostConv validation complete!")

In [ ]:
# CoordAtt Validation
print("=" * 60)
print("Coordinate Attention Validation")
print("=" * 60)

from src.blocks import CoordAtt, HSigmoid, HSwish

# Test configurations
configs = [
    {"name": "Basic (64→64)", "params": {"inp": 64, "oup": 64}, "input_shape": (2, 64, 56, 56)},
    {"name": "With reduction=16", "params": {"inp": 128, "oup": 128, "reduction": 16}, "input_shape": (2, 128, 28, 28)},
    {"name": "Rectangular input", "params": {"inp": 64, "oup": 64}, "input_shape": (2, 64, 56, 28)},
]

for cfg in configs:
    coord_att = CoordAtt(**cfg['params'])
    x = torch.randn(*cfg['input_shape'])
    
    # Get expected output shape (same as input for CoordAtt)
    expected_shape = cfg['input_shape']
    
    success = validate_block(
        coord_att, 
        cfg['input_shape'], 
        expected_shape,
        cfg['name']
    )
    
    # Show internal mip value
    print(f"  Internal: mip={coord_att.mip}")
    
    # Visualize attention maps
    coord_att.eval()
    with torch.no_grad():
        a_h, a_w = coord_att.get_attention_maps(x)
    print(f"  Attention shapes: a_h={a_h.shape}, a_w={a_w.shape}")

print("\n✅ Coordinate Attention validation complete!")

In [ ]:
# FusedInvertedResidualBlock Validation
print("=" * 60)
print("Fused Inverted Residual Block Validation")
print("=" * 60)

from src.blocks import FusedInvertedResidualBlock

# Test configurations
configs = [
    {"name": "With residual (64→64, s=1)", "params": {"inp": 64, "oup": 64, "stride": 1}, "input_shape": (2, 64, 56, 56)},
    {"name": "No residual (64→128, s=2)", "params": {"inp": 64, "oup": 128, "stride": 2}, "input_shape": (2, 64, 56, 56)},
    {"name": "With GhostConv", "params": {"inp": 64, "oup": 64, "stride": 1, "use_ghost": True}, "input_shape": (2, 64, 56, 56)},
    {"name": "expand_ratio=1", "params": {"inp": 64, "oup": 64, "expand_ratio": 1}, "input_shape": (2, 64, 56, 56)},
]

for cfg in configs:
    fused_ir = FusedInvertedResidualBlock(**cfg['params'])
    x = torch.randn(*cfg['input_shape'])
    
    # Calculate expected output shape
    stride = cfg['params'].get('stride', 1)
    oup = cfg['params']['oup']
    expected_h = cfg['input_shape'][2] // stride
    expected_w = cfg['input_shape'][3] // stride
    expected_shape = (cfg['input_shape'][0], oup, expected_h, expected_w)
    
    success = validate_block(
        fused_ir, 
        cfg['input_shape'], 
        expected_shape,
        cfg['name']
    )
    
    # Show residual info
    res_info = fused_ir.get_residual_info()
    print(f"  Residual: {res_info['use_res_connect']} ({res_info['reason']})")
    print(f"  Hidden dim: {fused_ir.hidden_dim}")

print("\n✅ Fused Inverted Residual Block validation complete!")

In [ ]:
# LinearDifferentialAttention Validation
print("=" * 60)
print("Linear Differential Attention (LDA) Validation")
print("=" * 60)

from src.blocks import LinearDifferentialAttention, NaiveFullAttention

# Test configurations
configs = [
    {"name": "Basic (D=256, heads=8)", "params": {"embed_dim": 256, "num_heads": 8}, "input_shape": (2, 196, 256)},
    {"name": "Small embed (D=128, heads=4)", "params": {"embed_dim": 128, "num_heads": 4}, "input_shape": (2, 49, 128)},
    {"name": "Large embed (D=512, heads=16)", "params": {"embed_dim": 512, "num_heads": 16}, "input_shape": (2, 100, 512)},
    {"name": "Single head", "params": {"embed_dim": 256, "num_heads": 1}, "input_shape": (2, 49, 256)},
]

for cfg in configs:
    lda = LinearDifferentialAttention(**cfg['params'])
    x = torch.randn(*cfg['input_shape'])
    
    # Expected shape is same as input for attention
    expected_shape = cfg['input_shape']
    
    success = validate_block(
        lda, 
        cfg['input_shape'], 
        expected_shape,
        cfg['name']
    )
    
    # Show internal configuration
    print(f"  Head dim: {lda.head_dim}, Scaling: {lda.scaling:.4f}")
    print(f"  Alpha: {lda.get_alpha():.4f}")

print("\n✅ Linear Differential Attention validation complete!")

In [ ]:
# PatchEmbedding & PositionalEncoding Validation
print("=" * 60)
print("Patch Embedding & Positional Encoding Validation")
print("=" * 60)

from src.blocks import PatchEmbedding, PositionalEncoding

# Test configurations for PatchEmbedding
print("\n--- Patch Embedding Tests ---")
pe_configs = [
    {"name": "Basic (64→256, p=4)", "params": {"in_channels": 64, "embed_dim": 256, "patch_size": 4}, 
     "input_shape": (2, 64, 56, 56), "expected_shape": (2, 196, 256)},
    {"name": "Small patch (p=2)", "params": {"in_channels": 64, "embed_dim": 256, "patch_size": 2}, 
     "input_shape": (2, 64, 14, 14), "expected_shape": (2, 49, 256)},
    {"name": "Large embed (D=512)", "params": {"in_channels": 64, "embed_dim": 512, "patch_size": 4}, 
     "input_shape": (2, 64, 56, 56), "expected_shape": (2, 196, 512)},
]

for cfg in pe_configs:
    patch_embed = PatchEmbedding(**cfg['params'])
    x = torch.randn(*cfg['input_shape'])
    
    success = validate_block(
        patch_embed, 
        cfg['input_shape'], 
        cfg['expected_shape'],
        cfg['name']
    )

# Test configurations for PositionalEncoding
print("\n--- Positional Encoding Tests ---")
pos_configs = [
    {"name": "Basic (N=196, D=256)", "params": {"embed_dim": 256}, "input_shape": (2, 196, 256)},
    {"name": "Short sequence (N=49)", "params": {"embed_dim": 256}, "input_shape": (2, 49, 256)},
    {"name": "Long sequence (N=784)", "params": {"embed_dim": 256}, "input_shape": (2, 784, 256)},
]

for cfg in pos_configs:
    pos_enc = PositionalEncoding(**cfg['params'])
    x = torch.randn(*cfg['input_shape'])
    
    success = validate_block(
        pos_enc, 
        cfg['input_shape'], 
        cfg['input_shape'],  # Output same as input
        cfg['name']
    )

print("\n✅ Patch Embedding & Positional Encoding validation complete!")

In [ ]:
# Positional Encoding Visualization
print("=" * 60)
print("Positional Encoding Visualization")
print("=" * 60)

import matplotlib.pyplot as plt

# Create positional encoding
pos_enc = PositionalEncoding(embed_dim=256, max_len=200)

# Get encoding for visualization
pe_matrix = pos_enc.visualize_encoding(seq_len=100)

# Plot heatmap
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Full heatmap
im1 = axes[0].imshow(pe_matrix.numpy(), cmap='RdBu', aspect='auto')
axes[0].set_title('Positional Encoding Heatmap')
axes[0].set_xlabel('Embedding Dimension')
axes[0].set_ylabel('Position')
plt.colorbar(im1, ax=axes[0])

# First few dimensions
axes[1].plot(pe_matrix[:, 0].numpy(), label='dim 0 (sin)')
axes[1].plot(pe_matrix[:, 1].numpy(), label='dim 1 (cos)')
axes[1].plot(pe_matrix[:, 2].numpy(), label='dim 2 (sin)')
axes[1].plot(pe_matrix[:, 3].numpy(), label='dim 3 (cos)')
axes[1].set_title('First 4 Dimensions of Positional Encoding')
axes[1].set_xlabel('Position')
axes[1].set_ylabel('Value')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Verify sinusoidal properties
print(f"\nSinusoidal Properties:")
print(f"  PE[0, 0] = {pe_matrix[0, 0].item():.4f} (should be sin(0) = 0)")
print(f"  PE[0, 1] = {pe_matrix[0, 1].item():.4f} (should be cos(0) = 1)")
print(f"  PE shape: {pe_matrix.shape}")
print(f"  Value range: [{pe_matrix.min():.4f}, {pe_matrix.max():.4f}]")

print("\n✅ Positional Encoding visualization complete!")

In [ ]:
# Combined Transition Pipeline
print("=" * 60)
print("Combined Transition Pipeline: CNN → Transformer")
print("=" * 60)

# Create pipeline components
patch_embed = PatchEmbedding(in_channels=64, embed_dim=256, patch_size=4)
pos_enc = PositionalEncoding(embed_dim=256, max_len=5000)

# Simulate CNN output
cnn_output = torch.randn(2, 64, 56, 56)

print(f"\n--- Shape Flow ---")
print(f"  CNN output:       {cnn_output.shape}")

# Apply patch embedding
patches = patch_embed(cnn_output)
print(f"  After PatchEmbed: {patches.shape}")
print(f"    - Num patches: {patch_embed.get_num_patches(56, 56)}")

# Apply positional encoding
transformer_input = pos_enc(patches)
print(f"  After PosEnc:     {transformer_input.shape}")

# Verify position info was added
diff = (transformer_input - patches).abs().mean()
print(f"\n  Position info added: {diff:.6f} mean diff")

# Parameter summary
pe_params = sum(p.numel() for p in patch_embed.parameters())
pos_params = sum(p.numel() for p in pos_enc.parameters())
print(f"\n--- Parameter Count ---")
print(f"  PatchEmbedding:     {pe_params:,}")
print(f"  PositionalEncoding: {pos_params:,} (sinusoidal buffer)")
print(f"  Total:              {pe_params + pos_params:,}")

print("\n✅ Transition pipeline validation complete!")

## 3. Template: Block Validation

Use this template for validating each block.

In [ ]:
def validate_block(block, input_shape, expected_output_shape, block_name):
    """
    Validate a block's functionality.
    
    Args:
        block: The neural network block to validate
        input_shape: Tuple of input dimensions
        expected_output_shape: Tuple of expected output dimensions
        block_name: Name of the block for display
    """
    print(f"\n{'='*60}")
    print(f"Validating: {block_name}")
    print(f"{'='*60}")
    
    # Create random input
    x = torch.randn(*input_shape)
    print(f"Input shape: {x.shape}")
    
    # Shape test
    try:
        check_output_shape(block, x, expected_output_shape)
        print(f"✅ Output shape correct: {expected_output_shape}")
    except AssertionError as e:
        print(f"❌ Shape error: {e}")
        return False
    
    # Gradient flow test
    grad_info = check_gradient_flow(block, x)
    all_grads = all(grad_info.values())
    if all_grads:
        print(f"✅ Gradients flow to all {len(grad_info)} parameters")
    else:
        missing = [k for k, v in grad_info.items() if not v]
        print(f"❌ Missing gradients for: {missing}")
    
    # NaN/Inf check
    block.eval()
    with torch.no_grad():
        output = block(x)
    try:
        check_no_nan_inf(output, "output")
        print(f"✅ No NaN/Inf in output")
    except AssertionError as e:
        print(f"❌ {e}")
        return False
    
    # Parameter count
    params = count_parameters(block)
    print(f"📊 Parameters: {params['total']:,} total, {params['trainable']:,} trainable")
    
    return True

# Example usage (will work once blocks are implemented):
# validate_block(GhostConv(3, 64), (2, 3, 224, 224), (2, 64, 224, 224), "GhostConv")

## 4. Visualization Helpers

In [ ]:
def visualize_feature_maps(feature_map, title="Feature Maps", num_channels=8):
    """
    Visualize feature maps from a convolutional layer.
    
    Args:
        feature_map: Tensor of shape (B, C, H, W)
        title: Title for the plot
        num_channels: Number of channels to display
    """
    if feature_map.dim() != 4:
        print("Expected 4D tensor (B, C, H, W)")
        return
    
    # Take first batch
    fm = feature_map[0].detach().cpu().numpy()
    
    # Limit channels
    num_channels = min(num_channels, fm.shape[0])
    
    fig, axes = plt.subplots(2, num_channels // 2, figsize=(12, 6))
    axes = axes.flatten()
    
    for i in range(num_channels):
        axes[i].imshow(fm[i], cmap='viridis')
        axes[i].set_title(f'Ch {i}')
        axes[i].axis('off')
    
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

def visualize_attention_map(attention, title="Attention Map"):
    """
    Visualize attention weights.
    
    Args:
        attention: Tensor of shape (B, heads, N, N) or (B, N, N)
        title: Title for the plot
    """
    attn = attention.detach().cpu().numpy()
    
    if attn.ndim == 4:
        # Average over heads
        attn = attn.mean(axis=1)
    
    # Take first batch
    attn = attn[0]
    
    plt.figure(figsize=(8, 8))
    plt.imshow(attn, cmap='hot')
    plt.colorbar()
    plt.title(title)
    plt.xlabel("Key Position")
    plt.ylabel("Query Position")
    plt.show()

print("Visualization helpers defined!")

## 5. Benchmark Results Template

In [ ]:
def run_benchmarks(blocks_config):
    """
    Run benchmarks for multiple blocks.
    
    Args:
        blocks_config: List of tuples (block, input_shape, name)
    
    Returns:
        Dictionary of benchmark results
    """
    results = {}
    
    for block, input_shape, name in blocks_config:
        print(f"\nBenchmarking: {name}")
        
        x = torch.randn(*input_shape)
        
        timing = warmup_and_benchmark(
            block, x, 
            num_runs=50, 
            warmup_runs=10, 
            device='cpu',  # Change to 'cuda' if available
            include_backward=True
        )
        
        results[name] = {
            'forward_ms': timing['forward_only_mean_ms'],
            'backward_ms': timing.get('backward_mean_ms', 0),
            'throughput': timing['throughput_samples_per_sec'],
            'params': count_parameters(block)['total']
        }
        
        print(f"  Forward: {timing['forward_only_mean_ms']:.3f} ms")
        print(f"  Backward: {timing.get('backward_mean_ms', 0):.3f} ms")
        print(f"  Throughput: {timing['throughput_samples_per_sec']:.1f} samples/sec")
    
    return results

# Will be used after blocks are implemented:
# benchmark_results = run_benchmarks([
#     (GhostConv(3, 64), (2, 3, 224, 224), "GhostConv"),
#     (FusedInvertedResidualBlock(64, 64), (2, 64, 56, 56), "FusedIR"),
#     # ... more blocks
# ])

print("Benchmark function defined!")

# LDA Attention Map Visualization
print("=" * 60)
print("LDA Attention Map Visualization")
print("=" * 60)

import matplotlib.pyplot as plt

# Create LDA and sample input
lda_viz = LinearDifferentialAttention(embed_dim=256, num_heads=8)
lda_viz.eval()

x_viz = torch.randn(1, 49, 256)  # 7x7 patches

with torch.no_grad():
    A1, A2, A_diff = lda_viz.get_attention_maps(x_viz)

# Plot attention maps for first head
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# A1 attention
im1 = axes[0].imshow(A1[0, 0].cpu().numpy(), cmap='hot')
axes[0].set_title('A1 (Attention Map 1)')
axes[0].set_xlabel('Key Position')
axes[0].set_ylabel('Query Position')
plt.colorbar(im1, ax=axes[0])

# A2 attention
im2 = axes[1].imshow(A2[0, 0].cpu().numpy(), cmap='hot')
axes[1].set_title('A2 (Attention Map 2)')
axes[1].set_xlabel('Key Position')
axes[1].set_ylabel('Query Position')
plt.colorbar(im2, ax=axes[1])

# Differential attention (A1 - A2)
diff_raw = (A1[0, 0] - A2[0, 0]).cpu().numpy()
im3 = axes[2].imshow(diff_raw, cmap='coolwarm', vmin=-0.1, vmax=0.1)
axes[2].set_title('A1 - A2 (Difference)')
axes[2].set_xlabel('Key Position')
axes[2].set_ylabel('Query Position')
plt.colorbar(im3, ax=axes[2])

# Final differential attention (α × (A1 - A2))
im4 = axes[3].imshow(A_diff[0, 0].cpu().numpy(), cmap='coolwarm')
axes[3].set_title(f'A_diff = α×(A1-A2), α={lda_viz.get_alpha():.2f}')
axes[3].set_xlabel('Key Position')
axes[3].set_ylabel('Query Position')
plt.colorbar(im4, ax=axes[3])

plt.tight_layout()
plt.suptitle('Linear Differential Attention Maps (Head 0)', y=1.02)
plt.show()

# Show attention statistics
print(f"\nAttention Map Statistics:")
print(f"  A1 range: [{A1.min():.4f}, {A1.max():.4f}]")
print(f"  A2 range: [{A2.min():.4f}, {A2.max():.4f}]")
print(f"  A_diff range: [{A_diff.min():.4f}, {A_diff.max():.4f}]")
print(f"  A1 row sum (should be 1.0): {A1[0, 0, 0].sum():.4f}")
print(f"  A2 row sum (should be 1.0): {A2[0, 0, 0].sum():.4f}")

print("\n✅ Attention visualization complete!")

In [ ]:
# LDA vs Naive Attention Comparison
print("=" * 60)
print("LDA vs Naive Full Attention Comparison")
print("=" * 60)

# Create both attention modules
lda_compare = LinearDifferentialAttention(embed_dim=256, num_heads=8)
naive_compare = NaiveFullAttention(embed_dim=256, num_heads=8)

lda_compare.eval()
naive_compare.eval()

# Test input
x_compare = torch.randn(2, 49, 256)

with torch.no_grad():
    y_lda = lda_compare(x_compare)
    y_naive = naive_compare(x_compare)

print(f"Input shape:       {x_compare.shape}")
print(f"LDA output shape:  {y_lda.shape}")
print(f"Naive output shape: {y_naive.shape}")

# Parameter comparison
lda_params = sum(p.numel() for p in lda_compare.parameters())
naive_params = sum(p.numel() for p in naive_compare.parameters())

print(f"\nParameter Count:")
print(f"  LDA:   {lda_params:,}")
print(f"  Naive: {naive_params:,}")
print(f"  Ratio: {lda_params/naive_params:.2f}x")

# Output statistics
print(f"\nOutput Statistics:")
print(f"  LDA   - mean: {y_lda.mean():.4f}, std: {y_lda.std():.4f}")
print(f"  Naive - mean: {y_naive.mean():.4f}, std: {y_naive.std():.4f}")

# Timing comparison
import time

def benchmark_module(module, x, num_runs=50):
    module.eval()
    # Warmup
    for _ in range(10):
        with torch.no_grad():
            _ = module(x)
    
    times = []
    for _ in range(num_runs):
        start = time.perf_counter()
        with torch.no_grad():
            _ = module(x)
        times.append((time.perf_counter() - start) * 1000)
    
    return sum(times) / len(times)

lda_time = benchmark_module(lda_compare, x_compare)
naive_time = benchmark_module(naive_compare, x_compare)

print(f"\nTiming (N=49, D=256):")
print(f"  LDA:   {lda_time:.3f} ms")
print(f"  Naive: {naive_time:.3f} ms")

print("\n✅ LDA vs Naive comparison complete!")

In [ ]:
# Alpha Parameter Behavior
print("=" * 60)
print("Alpha Parameter Analysis")
print("=" * 60)

import math

# Test different init values
init_values = [-1.0, 0.0, 0.5, 0.8, 1.0, 1.5]
print("Alpha initialization (α = exp(init)):")
print("-" * 40)

for init in init_values:
    lda_alpha = LinearDifferentialAttention(embed_dim=256, num_heads=8, init=init)
    expected = math.exp(init)
    actual = lda_alpha.get_alpha()
    print(f"  init={init:5.1f} → α={actual:.4f} (expected: {expected:.4f})")

# Verify alpha is learnable
print("\nAlpha gradient check:")
lda_grad = LinearDifferentialAttention(embed_dim=256, num_heads=8)
x_grad = torch.randn(2, 49, 256)

y_grad = lda_grad(x_grad)
loss = y_grad.sum()
loss.backward()

print(f"  Alpha value: {lda_grad.alpha.item():.4f}")
print(f"  Alpha gradient: {lda_grad.alpha.grad.item():.6f}")
print(f"  Alpha requires_grad: {lda_grad.alpha.requires_grad}")

print("\n✅ Alpha analysis complete!")